# Customer Churn Prediction with 5 ML Models

**Dataset:** Telco Customer Churn (public Kaggle dataset)  
**Task:** Binary classification — predict whether a customer will churn (`Yes` / `No`)  
**Models:** Logistic Regression, Decision Tree, KNN, Gaussian Naive Bayes, Random Forest


## 1. Imports


In [ ]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


## 2. Load dataset

Uses the local `archive/telco.csv` file so the notebook works on BITS Lab / local machines without a Colab-only Kaggle cache path.


In [ ]:
DATA_PATH = Path("archive/telco.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("telco.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 3. Cleaning and feature selection

Dropped columns:
- **IDs / constants:** `Customer ID`, `Country`, `State`, `Quarter`
- **High-cardinality location IDs:** `City`, `Zip Code`, `Latitude`, `Longitude`
- **Target leakage:** `Customer Status`, `Churn Score`, `Churn Category`, `Churn Reason`, `Satisfaction Score`

`Offer` and `Internet Type` nulls are filled with `None` (service not applicable).


In [ ]:
leakage_or_id_cols = [
    "Customer ID",
    "Customer Status",
    "Churn Score",
    "Churn Category",
    "Churn Reason",
    "Satisfaction Score",
    "Country",
    "State",
    "Quarter",
    "City",
    "Zip Code",
    "Latitude",
    "Longitude",
]

df = df.drop(columns=[c for c in leakage_or_id_cols if c in df.columns])

for col in ["Offer", "Internet Type"]:
    if col in df.columns:
        df[col] = df[col].fillna("None")

print("Missing values after cleaning:", int(df.isnull().sum().sum()))
print("Columns:", df.columns.tolist())


In [ ]:
y = df["Churn Label"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn Label"])

categorical_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Number of input features:", X.shape[1])
print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)
print("Churn rate:", round(y.mean(), 4))



## 4. Preprocessing and train/test split


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

# Dense encoding for GaussianNB (does not accept sparse matrices)
gnb_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


## 5. Helper: evaluate a model


In [ ]:
def evaluate_model(name, model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "AUC": roc_auc_score(y_te, y_prob),
        "Precision": precision_score(y_te, y_pred),
        "Recall": recall_score(y_te, y_pred),
        "F1": f1_score(y_te, y_pred),
        "MCC": matthews_corrcoef(y_te, y_pred),
    }

    print(f"\n========== {name} ==========\n")
    for k in ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]:
        print(f"{k:10s}: {metrics[k]:.4f}")

    print("\nConfusion Matrix\n")
    print(confusion_matrix(y_te, y_pred))
    print("\nClassification Report\n")
    print(classification_report(y_te, y_pred))
    return metrics, y_pred


## 6. Logistic Regression


In [ ]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
lr_model.fit(X_train, y_train)
lr_metrics, _ = evaluate_model("Logistic Regression", lr_model, X_test, y_test)


## 7. Decision Tree


In [ ]:
dt_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
    )),
])
dt_model.fit(X_train, y_train)
dt_metrics, _ = evaluate_model("Decision Tree", dt_model, X_test, y_test)


## 8. K-Nearest Neighbors


In [ ]:
knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=11)),
])
knn_model.fit(X_train, y_train)
knn_metrics, _ = evaluate_model("KNN", knn_model, X_test, y_test)


## 9. Gaussian Naive Bayes


In [ ]:
gnb_model = Pipeline([
    ("preprocessor", gnb_preprocessor),
    ("classifier", GaussianNB()),
])
gnb_model.fit(X_train, y_train)
gnb_metrics, _ = evaluate_model("Gaussian Naive Bayes", gnb_model, X_test, y_test)


## 10. Random Forest (Ensemble)


In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])
rf_model.fit(X_train, y_train)
rf_metrics, _ = evaluate_model("Random Forest", rf_model, X_test, y_test)


## 11. Model comparison table


In [ ]:
results_df = pd.DataFrame([
    lr_metrics,
    dt_metrics,
    knn_metrics,
    gnb_metrics,
    rf_metrics,
])

display_cols = ["Model", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
comparison = results_df[display_cols].copy()
for col in display_cols[1:]:
    comparison[col] = comparison[col].round(4)

print("\n===== Comparison Table =====\n")
print(comparison.to_string(index=False))

best_row = results_df.loc[results_df["F1"].idxmax()]
print(
    f"\nOverall winner (by F1): {best_row['Model']} "
    f"(F1={best_row['F1']:.4f}, AUC={best_row['AUC']:.4f}, MCC={best_row['MCC']:.4f})"
)


### Observations

| ML Model | Observation |
|---|---|
| Logistic Regression | Best overall on this hold-out set (highest F1, AUC, and MCC). Strong calibrated linear baseline after scaling + one-hot encoding. |
| Decision Tree | Interpretable; constrained depth reduces overfit, but slightly behind LR/RF on F1 and AUC. |
| KNN | Competitive after StandardScaler; still sensitive to class imbalance and feature space size. |
| Gaussian Naive Bayes | Highest Recall but lowest Precision — flags many churners, with more false alarms. Independence assumption limits F1/MCC. |
| Random Forest | Tied-best Accuracy and strong Precision/AUC; slightly behind LR on F1/Recall for the churn class. |
| **Overall Winner** | **Logistic Regression** (best F1 / AUC / MCC on the test set). |



## 12. Save models and test data (for Streamlit)


In [ ]:
os.makedirs("model", exist_ok=True)

joblib.dump(lr_model, "model/logistic_regression.joblib")
joblib.dump(dt_model, "model/decision_tree.joblib")
joblib.dump(knn_model, "model/knn.joblib")
joblib.dump(gnb_model, "model/naive_bayes.joblib")
joblib.dump(rf_model, "model/random_forest.joblib")

# Test CSV includes features + true label for Streamlit evaluation
test_export = X_test.copy()
test_export["Churn Label"] = y_test.map({1: "Yes", 0: "No"}).values
test_export.to_csv("test_data.csv", index=False)

comparison.to_csv("model/metrics_comparison.csv", index=False)

print("Saved models to model/")
print("Saved test_data.csv with", len(test_export), "rows")
print("Saved model/metrics_comparison.csv")
